In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,BatchNormalization, Activation
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

In [3]:
data = pd.read_csv("/kaggle/input/datasets/ankushpanday1/alzheimers-prediction-dataset-global/alzheimers_prediction_dataset.csv")
data.head(5)

,Country,Age,Gender,Education Level,BMI,Physical Activity Level,Smoking Status,Alcohol Consumption,Diabetes,Hypertension,...,Dietary Habits,Air Pollution Exposure,Employment Status,Marital Status,Genetic Risk Factor (APOE-ε4 allele),Social Engagement Level,Income Level,Stress Levels,Urban vs Rural Living,Alzheimer’s Diagnosis
0,Spain,90,Male,1,33.0,Medium,Never,Occasionally,No,No,...,Healthy,High,Retired,Single,No,Low,Medium,High,Urban,No
1,Argentina,72,Male,7,29.9,Medium,Former,Never,No,No,...,Healthy,Medium,Unemployed,Widowed,No,High,Low,High,Urban,No
2,South Africa,86,Female,19,22.9,High,Current,Occasionally,No,Yes,...,Average,Medium,Employed,Single,No,Low,Medium,High,Rural,No
3,China,53,Male,17,31.2,Low,Never,Regularly,Yes,No,...,Healthy,Medium,Retired,Single,No,High,Medium,Low,Rural,No
4,Sweden,58,Female,3,30.0,High,Former,Never,Yes,No,...,Unhealthy,High,Employed,Married,No,Low,Medium,High,Rural,No


In [4]:
print(f"shape of data is {data.shape}")
print(f"Columns are {data.columns}")

shape of data is (74283, 25)
Columns are Index(['Country', 'Age', 'Gender', 'Education Level', 'BMI',
       'Physical Activity Level', 'Smoking Status', 'Alcohol Consumption',
       'Diabetes', 'Hypertension', 'Cholesterol Level',
       'Family History of Alzheimer’s', 'Cognitive Test Score',
       'Depression Level', 'Sleep Quality', 'Dietary Habits',
       'Air Pollution Exposure', 'Employment Status', 'Marital Status',
       'Genetic Risk Factor (APOE-ε4 allele)', 'Social Engagement Level',
       'Income Level', 'Stress Levels', 'Urban vs Rural Living',
       'Alzheimer’s Diagnosis'],
      dtype='object')


In [5]:
data['Cholesterol Level'].value_counts()

Cholesterol Level
Normal    51973
High      22310
Name: count, dtype: int64

In [6]:
data['Cognitive Test Score'].value_counts()

Cognitive Test Score
62    1149
82    1122
45    1120
80    1114
49    1109
      ... 
58    1018
37    1016
48    1014
39    1010
42    1009
Name: count, Length: 70, dtype: int64

In [7]:
data.isnull().sum()

Country                                 0
Age                                     0
Gender                                  0
Education Level                         0
BMI                                     0
Physical Activity Level                 0
Smoking Status                          0
Alcohol Consumption                     0
Diabetes                                0
Hypertension                            0
Cholesterol Level                       0
Family History of Alzheimer’s           0
Cognitive Test Score                    0
Depression Level                        0
Sleep Quality                           0
Dietary Habits                          0
Air Pollution Exposure                  0
Employment Status                       0
Marital Status                          0
Genetic Risk Factor (APOE-ε4 allele)    0
Social Engagement Level                 0
Income Level                            0
Stress Levels                           0
Urban vs Rural Living             

In [8]:
data.duplicated().sum()

np.int64(0)

In [9]:
data.describe(include='all')

,Country,Age,Gender,Education Level,BMI,Physical Activity Level,Smoking Status,Alcohol Consumption,Diabetes,Hypertension,...,Dietary Habits,Air Pollution Exposure,Employment Status,Marital Status,Genetic Risk Factor (APOE-ε4 allele),Social Engagement Level,Income Level,Stress Levels,Urban vs Rural Living,Alzheimer’s Diagnosis
count,74283,74283.000000,74283,74283.000000,74283.000000,74283,74283,74283,74283,74283,...,74283,74283,74283,74283,74283,74283,74283,74283,74283,74283
unique,20,NaN,2,NaN,NaN,3,3,3,2,2,...,3,3,3,3,2,3,3,3,2,2
top,Brazil,NaN,Female,NaN,NaN,High,Current,Never,No,No,...,Average,High,Unemployed,Single,No,Medium,Low,Medium,Rural,No
freq,3839,NaN,37249,NaN,NaN,24853,24915,24865,59527,52134,...,24917,24906,24801,25169,59561,24859,24873,24886,37203,43570
mean,NaN,71.964703,NaN,9.487514,26.780639,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,12.980748,NaN,5.757020,4.764679,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,50.000000,NaN,0.000000,18.500000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,61.000000,NaN,4.000000,22.700000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,72.000000,NaN,9.000000,26.800000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,83.000000,NaN,14.000000,30.900000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
data.rename({'Alzheimer’s Diagnosis': 'Alzheimer_Diagnosis'},inplace=True)

In [11]:
target = [ 'Alzheimer’s Diagnosis']

In [12]:
import json

selected_features= [
    'Age', 
    'Genetic Risk Factor (APOE-ε4 allele)', 
    'Family History of Alzheimer’s', 
    'Education Level',
    'Income Level', 
    'Urban vs Rural Living', 
    'Stress Levels',
    'Smoking Status',
    'Employment Status',
    'Air Pollution Exposure',
    'Alzheimer’s Diagnosis'
]

data = data[selected_features]

data.head(5)

,Age,Genetic Risk Factor (APOE-ε4 allele),Family History of Alzheimer’s,Education Level,Income Level,Urban vs Rural Living,Stress Levels,Smoking Status,Employment Status,Air Pollution Exposure,Alzheimer’s Diagnosis
0,90,No,No,1,Medium,Urban,High,Never,Retired,High,No
1,72,No,No,7,Low,Urban,High,Former,Unemployed,Medium,No
2,86,No,No,19,Medium,Rural,High,Current,Employed,Medium,No
3,53,No,No,17,Medium,Rural,Low,Never,Retired,Medium,No
4,58,No,No,3,Medium,Rural,High,Former,Employed,High,No


In [13]:
jsondata = []
for _, row in data.iterrows():
    features = row.drop(target).to_dict()
    jsondata.append({
        "input": features,
        "prediction": row[target].item()
    })

In [14]:
jsondata[74000]

{'input': {'Age': 60,
  'Genetic Risk Factor (APOE-ε4 allele)': 'No',
  'Family History of Alzheimer’s': 'No',
  'Education Level': 17,
  'Income Level': 'Low',
  'Urban vs Rural Living': 'Rural',
  'Stress Levels': 'Medium',
  'Smoking Status': 'Former',
  'Employment Status': 'Unemployed',
  'Air Pollution Exposure': 'Low'},
 'prediction': 'No'}

In [16]:
test_label = [item['prediction'] for item in jsondata]

In [15]:
with open("/kaggle/working/data.json","w") as file:
    json.dump(jsondata,file,indent=2)

In [16]:
with open("/kaggle/working/data.json","r") as file:
    data = json.load(file)

In [17]:
rows = []
for patient in data:
    text = (
        f"The patient's age is {patient['input']['Age']}."
        f"The patient's Genetic Risk Factor (APOE-ε4 allele) is {patient['input']['Genetic Risk Factor (APOE-ε4 allele)']}."
        f"The patient's Family History of Alzheimer’s is {patient['input']['Family History of Alzheimer’s']}."
        f"The patient's Education Level is {patient['input']['Education Level']}."
        f"The patient's Income Level is {patient['input']['Income Level']}."
        f"The patient's Urban vs Rural Living is {patient['input']['Urban vs Rural Living']}."
        f"The patient's Stress Levels is {patient['input']['Stress Levels']}."
        f"The patient's Smoking Status is {patient['input']['Smoking Status']}."
        f"The patient's Employment Status is {patient['input']['Employment Status']}."
        f"The patient's Air Pollution Exposure is {patient['input']['Air Pollution Exposure']}."
    )

    target = patient['prediction']
    rows.append({"text": text,"Alzheimer’s Diagnosis": target})
    

In [18]:
rows[1]

{'text': "The patient's age is 72.The patient's Genetic Risk Factor (APOE-ε4 allele) is No.The patient's Family History of Alzheimer’s is No.The patient's Education Level is 7.The patient's Income Level is Low.The patient's Urban vs Rural Living is Urban.The patient's Stress Levels is High.The patient's Smoking Status is Former.The patient's Employment Status is Unemployed.The patient's Air Pollution Exposure is Medium.",
 'Alzheimer’s Diagnosis': 'No'}

In [19]:
data = pd.DataFrame(rows, columns=['text','Alzheimer’s Diagnosis'])

In [20]:
data.to_csv("alzhermer_text_data.csv",index=False)

In [21]:
data = pd.read_csv("/kaggle/working/alzhermer_text_data.csv")

data['text'][10]

"The patient's age is 80.The patient's Genetic Risk Factor (APOE-ε4 allele) is No.The patient's Family History of Alzheimer’s is No.The patient's Education Level is 2.The patient's Income Level is High.The patient's Urban vs Rural Living is Rural.The patient's Stress Levels is Low.The patient's Smoking Status is Current.The patient's Employment Status is Unemployed.The patient's Air Pollution Exposure is Medium."

In [22]:
data.head(4)

,text,Alzheimer’s Diagnosis
0,The patient's age is 90.The patient's Genetic ...,No
1,The patient's age is 72.The patient's Genetic ...,No
2,The patient's age is 86.The patient's Genetic ...,No
3,The patient's age is 53.The patient's Genetic ...,No


In [ ]:
pip uninstall transformers -y

In [ ]:
pip uninstall tokenizers -y

In [ ]:
pip uninstall huggingface-hub -y 

In [ ]:
pip install "transformers==5.0.0"

In [ ]:
pip install accelerate datasets torch

In [ ]:
pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [34]:
pip install trl  sentencepiece peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 4.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.8 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install torch torchvision

In [ ]:
pip install evaluate

In [23]:
! nvidia-smi

Thu Oct 22 09:21:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # only one GPU

In [25]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [35]:
import torch
import transformers
import accelerate
import trl
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
Accelerate: 1.14.0
TRL: 1.16.0
PEFT: 0.20.0


In [36]:
import transformers
from transformers import pipeline,set_seed
from transformers import DataCollatorWithPadding
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import re
from tqdm import tqdm
import torch

In [ ]:
data.head(10)

In [38]:
from datasets import Dataset
dataset = Dataset.from_pandas(data)

In [39]:
dataset= dataset.train_test_split(test_size=0.3,seed=42)

In [40]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'Alzheimer’s Diagnosis'],
        num_rows: 51998
    })
    test: Dataset({
        features: ['text', 'Alzheimer’s Diagnosis'],
        num_rows: 22285
    })
})

In [30]:
dataset['train'].column_names

['text', 'Alzheimer’s Diagnosis']

In [31]:
dataset['train'][1]

{'text': "The patient's age is 87.The patient's Genetic Risk Factor (APOE-ε4 allele) is No.The patient's Family History of Alzheimer’s is Yes.The patient's Education Level is 9.The patient's Income Level is High.The patient's Urban vs Rural Living is Rural.The patient's Stress Levels is High.The patient's Smoking Status is Never.The patient's Employment Status is Unemployed.The patient's Air Pollution Exposure is Low.",
 'Alzheimer’s Diagnosis': 'No'}

In [39]:
dataset['train']['text'][5]

"The patient's age is 58.The patient's Genetic Risk Factor (APOE-ε4 allele) is No.The patient's Family History of Alzheimer’s is No.The patient's Education Level is 1.The patient's Income Level is High.The patient's Urban vs Rural Living is Rural.The patient's Stress Levels is Low.The patient's Smoking Status is Current.The patient's Employment Status is Retired.The patient's Air Pollution Exposure is Low."

In [41]:
model_name = "distilbert/distilbert-base-uncased"  # Example model
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [42]:
from transformers import BitsAndBytesConfig


model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 2,
    id2label = {0:"No",1:"Yes"},
    label2id = {"No":0, "Yes":1}
)

print("Model Loaded Successfully")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded Successfully


In [43]:
print(model.config.num_labels)
print(model.config.id2label)
print(model.classifier)

2
{0: 'No', 1: 'Yes'}
Linear(in_features=768, out_features=2, bias=True)


# Label Encoding
# Convert Target into Numbers

In [44]:
dataset['train']['Alzheimer’s Diagnosis'][1]

'No'

In [45]:
def text_token_function(examples):
    texts = [
        text if text is not None else ""
        for text in examples['text']
    ]

    input_encoding =tokenizer(texts,truncation = True,max_length = 256,padding= "max_length")
    labels = [str(label).strip().lower()
             for label in examples['Alzheimer’s Diagnosis']
            ]
    if not all(label in {"yes", "no"} for label in labels):
        raise ValueError(f"Unexpected labels: {labels}")

    return{
         "input_ids": input_encoding["input_ids"],
         "attention_mask": input_encoding["attention_mask"],
         "label": [1 if label == "yes" else 0
                   for label in labels
                  ]
    }

In [46]:
train = dataset['train'].map(text_token_function,batched=True,remove_columns = ['Alzheimer’s Diagnosis','text'])
test = dataset['test'].map(text_token_function,batched=True,remove_columns = ['Alzheimer’s Diagnosis','text'])

Map:   0%|          | 0/51998 [00:00<?, ? examples/s]

Map:   0%|          | 0/22285 [00:00<?, ? examples/s]

In [47]:
from collections import Counter

print(Counter(train["label"]))

Counter({0: 30351, 1: 21647})


In [48]:
train[1]

{'input_ids': [101,
  1996,
  5776,
  1005,
  1055,
  2287,
  2003,
  6584,
  1012,
  1996,
  5776,
  1005,
  1055,
  7403,
  3891,
  5387,
  1006,
  9706,
  8913,
  1011,
  1159,
  2549,
  2035,
  12260,
  1007,
  2003,
  2053,
  1012,
  1996,
  5776,
  1005,
  1055,
  2155,
  2381,
  1997,
  21901,
  1521,
  1055,
  2003,
  2748,
  1012,
  1996,
  5776,
  1005,
  1055,
  2495,
  2504,
  2003,
  1023,
  1012,
  1996,
  5776,
  1005,
  1055,
  3318,
  2504,
  2003,
  2152,
  1012,
  1996,
  5776,
  1005,
  1055,
  3923,
  5443,
  3541,
  2542,
  2003,
  3541,
  1012,
  1996,
  5776,
  1005,
  1055,
  6911,
  3798,
  2003,
  2152,
  1012,
  1996,
  5776,
  1005,
  1055,
  9422,
  3570,
  2003,
  2196,
  1012,
  1996,
  5776,
  1005,
  1055,
  6107,
  3570,
  2003,
  18787,
  1012,
  1996,
  5776,
  1005,
  1055,
  2250,
  10796,
  7524,
  2003,
  2659,
  1012,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
 

In [49]:
print("Tokenizer:", tokenizer.name_or_path)
print("Padding token:", tokenizer.pad_token)
print("Padding token ID:", tokenizer.pad_token_id)
print("Padding side:", tokenizer.padding_side)

Tokenizer: distilbert/distilbert-base-uncased
Padding token: [PAD]
Padding token ID: 0
Padding side: right


In [50]:
from peft import LoraConfig,get_peft_model
lora_config = LoraConfig(
    r = 16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    lora_alpha = 16,    # scaling factor
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
)
model= get_peft_model(model,lora_config)
model.print_trainable_parameters()

ValueError: Target modules {'o_proj', 'v_proj', 'q_proj', 'k_proj'} not found in the base model. Please check the target modules and try again.

In [51]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score)

def compute_metrics(eval_pred):
    logits,labels = eval_pred
    predictions = np.argmax(logits,axis=-1)
    precision,recall,f1,_ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division = 0
        )
        
    )
    #probabilities = torch.softmax(torch.tensor(logits),dim=1).numpy().cpu()[:,1]
    metrics =  {
        "accuracy": accuracy_score(labels,predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1 
    }
    #if len(np.unique(labels))==2:
        #metrics['roc_auc'] =  roc_auc_score(labels,probabilities)

    return  metrics

In [52]:
train_args = TrainingArguments(
    output_dir = '/result',                       # where model checkpoints will be saved

    num_train_epochs = 50,                        # ❌ too low (use 2–5 for better summarization)
    optim = "adamw_8bit",

    per_device_train_batch_size = 8,              # batch size for training (Kaggle-safe)

    per_device_eval_batch_size = 8,               # batch size for evaluation
    gradient_accumulation_steps = 2,

    weight_decay = 0.01,                          # regularization to reduce overfitting
    

    fp16 = torch.cuda.is_available(),

    eval_strategy="epoch",                         # ⚠️ should be evaluation_strategy in newer versions
    save_strategy="epoch",

    eval_steps =100,                               # run evaluation every 100 steps

    #save_steps =100,                               # save model every 100 steps
    logging_strategy = "steps",
    logging_steps = 100,

    learning_rate = 2e-5,                         # good learning rate for mT5 fine-tuning


    metric_for_best_model ="f1",

    warmup_steps = 200,                              # helps stabilize early training
    report_to="none",

                                                     # or specifically: report_to="none"



)

In [53]:
print(train.column_names)

['input_ids', 'attention_mask', 'label']


In [54]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset = train,
    eval_dataset = test,
    data_collator = DataCollatorWithPadding(tokenizer),
    args = train_args,
    compute_metrics = compute_metrics
)

In [55]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.095582,0.536436,0.720260,0.643990,0.698544,0.670159
2,1.051478,0.540371,0.719677,0.631741,0.745533,0.683936
3,1.075294,0.541210,0.721023,0.638503,0.724355,0.678725
4,1.042188,0.539169,0.721023,0.638503,0.724355,0.678725
5,1.050907,0.539151,0.721023,0.638503,0.724355,0.678725
6,1.036180,0.541162,0.721023,0.638503,0.724355,0.678725
7,1.066258,0.541322,0.720081,0.647138,0.685970,0.665988
8,1.102508,0.537468,0.721023,0.638503,0.724355,0.678725
9,1.081683,0.540191,0.721023,0.646088,0.694904,0.669607
10,1.041451,0.539277,0.721023,0.638503,0.724355,0.678725


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [65]:
import numpy as np
from collections import Counter

pred_output = trainer.predict(train)

logits = pred_output.predictions
labels = pred_output.label_ids

predictions = np.argmax(logits, axis=-1)

print("Actual labels:", Counter(labels))
print("Predicted labels:", Counter(predictions))
print("First 20 actual:", labels[:20])
print("First 20 predictions:", predictions[:20])

Actual labels: Counter({np.int64(0): 30351, np.int64(1): 21647})
Predicted labels: Counter({np.int64(0): 51998})
First 20 actual: [1 0 0 0 1 0 1 1 0 0 1 1 0 1 0 0 0 0 0 1]
First 20 predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
